# Energy signature

The **energy signature** is the simplest and most used way to characterise how a building uses energy: plot the **daily energy** against the **daily mean outdoor temperature** and fit a line

$$E = a + b\,T_{out}$$

- **slope b** (kWh/day per degC; negative for heating): how strongly the consumption reacts to the weather. Divided by 24 h it is, roughly, the overall heat-loss coefficient of the building (envelope + ventilation) in kW/K, divided by the efficiency of the system;
- **intercept a** (kWh/day): the consumption extrapolated at 0 degC; the *base load* is the consumption that does not depend on the weather (appliances, lighting, hot water);
- **heating balance temperature**: above some outdoor temperature the heating is off and the consumption becomes flat (the *change point*). A single line fits only the heating season; a change-point model $E = a + b\,\max(0, T_b - T_{out})$ also catches the flat part;
- **R^2** says how much of the daily variation the weather explains. Occupancy, solar gains and operation not related to the weather lower it.

**Why the baseline?** The signature describes the building as it is operated by its native control: it is the reference against which a controller is judged (the controlled run should lie below the baseline line for the same temperature).

**What you need in your data**
1. a **daily energy** series of the baseline, in the schema you wrote in the previous notebook;
2. an **outdoor air temperature** series stored by your subscriber. This means that among the devices you chose in *Sensors & actuators* there must be **Site Outdoor Air Drybulb Temperature**. If you do not have it: go back to *Sensors & actuators*, add it, re-export the FMU, re-run the building (baseline) and make sure your gateway publishes it and your subscriber stores it.

**How many points?** One point per simulated day. A 7-day simulation gives 7 points: a regression on 7 points is a toy. Simulate a longer period (the simulated period is set in the *Control room*; a month in winter, or better a whole heating season, with varied weather) before drawing conclusions.

In [ ]:
import base64, json, os, urllib.error, urllib.parse, urllib.request
import numpy as np
import pandas as pd

INFLUX_URL  = os.getenv("INFLUX_URL", "http://influxdb:8086")
INFLUX_USER = os.getenv("INFLUX_USERNAME", "admin")
INFLUX_PASS = os.getenv("INFLUX_PASSWORD", "")

DB_BASE = "base_building"        # baseline run: native EnergyPlus control
DB_CTRL = "controlled_building"  # controlled run: your BMS closes the loop


def query(db, influxql):
    """Run one InfluxQL statement on database `db` and return a pandas DataFrame.

    - SELECT: one row per point; the `time` column is a UTC datetime; tags of a
      GROUP BY are added as columns; several series are concatenated.
    - SHOW ...: the plain table returned by InfluxDB.
    - empty result: an empty DataFrame (check `df.empty`).
    Only read statements work here (HTTP GET).
    """
    params = urllib.parse.urlencode({"db": db, "q": influxql, "epoch": "ms"})
    req = urllib.request.Request(f"{INFLUX_URL}/query?{params}")
    token = base64.b64encode(f"{INFLUX_USER}:{INFLUX_PASS}".encode()).decode()
    req.add_header("Authorization", "Basic " + token)
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            body = json.load(resp)
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"InfluxDB answered HTTP {e.code}: {e.read().decode(errors='replace')[:300]}") from None
    except urllib.error.URLError as e:
        raise RuntimeError(f"Cannot reach InfluxDB at {INFLUX_URL}: {e.reason}") from None
    frames = []
    for result in body.get("results", []):
        if "error" in result:
            raise RuntimeError(f"InfluxQL error in database '{db}': {result['error']}")
        for series in result.get("series", []):
            df = pd.DataFrame(series.get("values", []), columns=series["columns"])
            for tag, value in (series.get("tags") or {}).items():
                df[tag] = value
            if "time" in df.columns:
                df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
import matplotlib.pyplot as plt

def plot_signature(T, E, a=None, b=None, title="Energy signature", extra_line=None):
    """Scatter of daily energy E (kWh/day) vs daily mean outdoor temperature T (degC).
    If a and b are given, the line E = a + b*T is drawn too; extra_line=(x, y) draws any other curve."""
    T, E = np.asarray(T, dtype=float), np.asarray(E, dtype=float)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.scatter(T, E, color="tab:blue", label="days")
    if a is not None and b is not None:
        x = np.linspace(T.min(), T.max(), 50)
        ax.plot(x, a + b * x, color="tab:red", label=f"E = {a:.1f} + ({b:.2f}) T")
    if extra_line is not None:
        ax.plot(*extra_line, color="tab:green", linestyle="--", label="change-point model")
    ax.set_xlabel("daily mean outdoor temperature (degC)"); ax.set_ylabel("daily energy (kWh/day)")
    ax.set_title(title); ax.grid(alpha=.3); ax.legend()
    return ax

daily_energy = None   # you will fill: pandas Series, index = day, values = kWh/day
daily_temp = None     # you will fill: pandas Series, index = day, values = degC

## 1. Daily energy of the baseline

**TODO.** Reuse what you did in the *data access* notebook: daily energy of the baseline in kWh/day, as a pandas Series indexed by day (UTC). Mind the end-of-step time stamp and the increment-or-cumulative question. Print it.

In [ ]:
# TODO: daily energy (kWh/day) of the baseline -> daily_energy
raise NotImplementedError("TODO: daily_energy = ...")

## 2. Daily mean outdoor temperature

**TODO.** From the **outdoor drybulb temperature that your subscriber stored**, compute the daily mean (degC) as a Series indexed by day, on the **same days** as the energy. If your schema stores it as a point/tag, find it with `SHOW TAG VALUES`; if you cannot find any outdoor temperature, read the introduction again: you must add the sensor, not look for a substitute.

In [ ]:
# TODO: daily mean outdoor drybulb temperature (degC) from YOUR database -> daily_temp
raise NotImplementedError("TODO: daily_temp = ...  (needs the outdoor temperature sensor among your devices)")

## 3. Fit the regression

**TODO.** Build a table with one row per day (`T`, `E`), fit `E = a + b*T` with `numpy.polyfit(T, E, 1)` (or scikit-learn `LinearRegression`) and compute `R^2 = 1 - SS_res / SS_tot`. Print `a`, `b`, `R^2` and the number of points.

In [ ]:
# TODO: fit E = a + b*T and compute R^2
raise NotImplementedError("TODO: a, b, r2 = ...")

## 4. Plot
Scatter of the days with the fitted line (`plot_signature` is given).

In [ ]:
# TODO: plot_signature(T, E, a, b)
raise NotImplementedError("TODO: call plot_signature with your data and your a, b")

## 5. Optional: heating balance temperature (change point)

Fit $E = a + b\,\max(0, T_b - T)$ by scanning the balance temperature $T_b$ and keeping the one with the smallest residual (for each $T_b$ the model is linear in $a, b$). With few days the result is not reliable: use it with a long simulation.

In [ ]:
# OPTIONAL TODO: change-point model (scan Tb, fit E = a + b*max(0, Tb - T))
print("optional: not implemented")

## 6. Interpretation (fill in)

*TODO.* In a few sentences, using your own numbers:
- what are the sign and the order of magnitude of the slope `b` (kWh/day per degC)? What does it tell about the building?
- what does the intercept `a` mean here, and is it a realistic base load?
- what does R^2 tell you? Which other factors (solar gains, occupancy, weekends) explain the rest?
- how many days did you use? How much would you trust the result, and what would you do to improve it (longer period, change point)?
- where would the controlled run lie on this plot if the controller works?